# 03 · Codebook IEFIC — Encuesta de Ingresos y Gastos (BANREP)

**Base de datos:** 3 de 3 — IEFIC 2017-2018 (Banco de la República) · archivo `BANREP-IEFIC-2017-2018.xml` (diccionario de variables DDI de la Encuesta de Ingresos y Gastos de los Hogares).
**Nota:** este archivo es un **codebook** (diccionario de variables), no contiene microdatos. Documenta 331 variables de la encuesta.

**Utilidad para el proyecto:** la IEFIC es la fuente oficial de **ingresos y gastos de los hogares colombianos** — insumo clave para caracterizar la capacidad de pago de los solicitantes de crédito.

In [1]:
import sys
import pandas as pd

sys.path.insert(0, "..")
from src.utils.loaders import load_iefic_codebook

cb = load_iefic_codebook()
print(f"Variables únicas: {cb.shape[0]}")
cb.head()

Variables únicas: 331


,name,label,question,type
0,SECUENCIA_P,secuencia_p,SECUENCIA_P,discrete
1,ORDEN,orden,ORDEN,discrete
2,DIRECTORIO,DIRECTORIO,,contin
3,INGRESO_COMPLETO,"estado del ingreso total 1=completo, 0=falta a...","Estado del ingreso total 1=completo, 0=Falta a...",discrete
4,P6050,¿cuál es el parentesco de ... con el jefe o je...,¿Cuál es el parentesco de ... Con el jefe o je...,discrete


In [2]:
print("Tipo de variable:")
print(cb["type"].value_counts().to_string())
print("\nVariables con pregunta documentada:", cb["question"].str.len().gt(0).sum())

Tipo de variable:
type
discrete    206
contin      125

Variables con pregunta documentada: 328


## Limpieza y normalización del XML (codebook DDI)

A diferencia de los otros dos datasets, aquí la fuente no es un CSV sino un **XML en estándar DDI** (*Data Documentation Initiative*), el estándar internacional para documentar encuestas sociales y económicas (namespace `http://www.icpsr.umich.edu/DDI`). El Banco de la República publica en ese formato el **codebook** (diccionario de variables) de la IEFIC.

La limpieza que hace `load_iefic_codebook` (`src/utils/loaders.py`) es una **extracción estructurada**: recorrer los nodos `<var>` del XML y convertirlos en un DataFrame tabular, con una deduplicación final. Veamos cada paso contra el archivo crudo.

In [3]:
from src.utils.loaders import RAW
from lxml import etree

arbol = etree.parse(str(RAW / "BANREP-IEFIC-2017-2018.xml"))
ns = {"ddi": "http://www.icpsr.umich.edu/DDI"}

nodos_var = arbol.xpath("//ddi:var", namespaces=ns)
print("Nodos <var> en el XML crudo :", len(nodos_var))
print("Nombres de variable únicos  :", len({v.get("name") for v in nodos_var}))
archivos = [a.strip() for a in arbol.xpath("//ddi:fileTxt/ddi:fileName/text()", namespaces=ns)]
print("Archivos declarados         :", archivos)

print("\nAsí se ve una variable en el estándar DDI (fragmento):")
print(etree.tostring(nodos_var[0], pretty_print=True, encoding="unicode")[:500])

Nodos <var> en el XML crudo : 662
Nombres de variable únicos  : 331
Archivos declarados         : ['IEFIC_2017.NSDstat', 'IEFIC_2018.NSDstat']

Así se ve una variable en el estándar DDI (fragmento):
<var xmlns="http://www.icpsr.umich.edu/DDI" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" ID="V5015" name="SECUENCIA_P" files="F18" dcml="0" intrvl="discrete">
      <location StartPos="1" EndPos="1" width="1" RecSegNo="1"/>
      <labl>
        secuencia_p
      </labl>
      <security>
        El acceso a microdatos y Mam-up se considera como de tratamiento especial respecto a la reserva estadística por tanto estará sujeto a la reglamentación que para el efecto defina el Comité de Aseg


### Extracción de campos y deduplicación

De cada nodo `<var>` se extraen cuatro campos:

| Elemento / atributo DDI | Columna resultante | Contenido |
|-------------------------|--------------------|-----------|
| atributo `name` de `<var>` | `name` | Nombre de la variable (ej. `P6050`, `INGTOTOB`) |
| `<labl>` | `label` | Etiqueta descriptiva corta |
| `<qstn>/<qstnLit>` | `question` | Literal de la pregunta en el cuestionario |
| atributo `intrvl` | `type` | Tipo de variable: `discrete` (categórica/entera) o `contin` (continua) |

**¿Por qué hace falta deduplicar?** El codebook describe el conjunto de variables **una vez por archivo de la encuesta**, y este XML contiene dos: `IEFIC_2017.NSDstat` (F17) e `IEFIC_2018.NSDstat` (F18). Como ambas ediciones comparten el cuestionario, cada variable aparece dos veces: **662 nodos `<var>` para 331 variables reales**. La regla `drop_duplicates(subset="name", keep="first")` conserva la primera definición de cada una.

In [4]:
# Verificación: filas antes y después de la deduplicación
nombres_crudo = pd.Series([v.get("name") for v in nodos_var])

print("Filas ANTES del drop_duplicates :", len(nombres_crudo))
print("Filas DESPUÉS (DataFrame cb)    :", cb.shape[0])
print("Duplicados eliminados           :", len(nombres_crudo) - cb.shape[0])

frecuencias = nombres_crudo.value_counts()
print("\nFrecuencia de cada nombre en el XML:", frecuencias.value_counts().to_dict())

print("\nTipos de variable en el crudo    :",
      pd.Series([v.get("intrvl") for v in nodos_var]).value_counts().to_dict())
print("Tipos tras la deduplicación (cb) :", cb["type"].value_counts().to_dict())

print("\nResultado final:", cb.shape, "| nulos:", int(cb.isna().sum().sum()),
      "| duplicados:", int(cb.duplicated().sum()))

Filas ANTES del drop_duplicates : 662
Filas DESPUÉS (DataFrame cb)    : 331
Duplicados eliminados           : 331

Frecuencia de cada nombre en el XML: {2: 331}

Tipos de variable en el crudo    : {'discrete': 412, 'contin': 250}
Tipos tras la deduplicación (cb) : {'discrete': 206, 'contin': 125}



Resultado final: (331, 4) | nulos: 0 | duplicados: 0


### Conclusión de la limpieza

El XML crudo de 662 nodos `<var>` quedó convertido en un **diccionario tabular de 331 variables únicas** (4 columnas: `name`, `label`, `question`, `type`), sin duplicados y con textos limpios. Es la referencia oficial para interpretar las variables de la IEFIC cuando se analicen los ingresos, las deudas y la capacidad de pago de los hogares — no contiene microdatos, solo su documentación.

## Variables relevantes para riesgo de crédito

Buscamos variables de **ingresos, deuda, empleo y patrimonio** — los factores clásicos de capacidad de pago.

In [5]:
keywords = ["ingreso", "deuda", "crédito", "credito", "empleo", "trabajo", "patrimonio",
             "ahorro", "gasto", "ocupaci", "salario", "pension", "arriendo", "vivienda"]
mask = cb["label"].str.lower().str.contains("|".join(keywords), na=False) | \
       cb["question"].str.lower().str.contains("|".join(keywords), na=False)
relevantes = cb[mask].copy()
print(f"Variables potencialmente relevantes: {relevantes.shape[0]}")
relevantes[["name", "label", "type"]].head(30)

Variables potencialmente relevantes: 143


,name,label,type
3,INGRESO_COMPLETO,"estado del ingreso total 1=completo, 0=falta a...",discrete
6,INGTOTOB,ingreso total por persona,contin
10,P2439,¿algún miembro de este hogar es propietario de...,discrete
11,P2447,¿en qué año ud. o algún miembro de su hogar c...,contin
12,P2461,si usted quisiera vender esta vivienda ¿cuál s...,contin
13,P2462,"ud. o algún miembro de su hogar, ¿compró o con...",discrete
15,P2465,¿este subsidio correspondía a vivienda de int...,discrete
16,P2466,¿para la compra de esta vivienda utilizó crédi...,discrete
17,P2168,valor del crédito hipotecario $,contin
18,P2469,¿ud. o algún miembro del hogar conoce o conocí...,discrete


## Variables de ingreso total y componentes

In [6]:
ingreso = cb[cb["name"].str.contains("ING|P60|P61|P62|P63", na=False)]
ingreso[["name", "label", "type"]].head(20)

,name,label,type
3,INGRESO_COMPLETO,"estado del ingreso total 1=completo, 0=falta a...",discrete
4,P6050,¿cuál es el parentesco de ... con el jefe o je...,discrete
6,INGTOTOB,ingreso total por persona,contin
106,P622,¿………….tiene fondos mutuos o de inversión?,discrete


## Variables de deuda y crédito

In [7]:
deuda = cb[cb["label"].str.contains("deuda|crédito|credito|financi", case=False, na=False)]
deuda[["name", "label", "type"]].head(20)

,name,label,type
16,P2466,¿para la compra de esta vivienda utilizó crédi...,discrete
17,P2168,valor del crédito hipotecario $,contin
18,P2469,¿ud. o algún miembro del hogar conoce o conocí...,discrete
19,P2470,¿está pagando este crédito hipotecario actualm...,discrete
20,P2471_1,¿cuánto dinero paga (o debería estar pagando a...,contin
21,P2471_2,¿cuánto dinero paga (o debería estar pagando a...,contin
22,P2471_3,¿cuánto dinero paga (o debería estar pagando a...,contin
23,P2471_4,¿cuánto dinero paga (o debería estar pagando a...,contin
24,P2472,¿con qué tipo de institución tomó el crédito h...,discrete
26,P2473,ud. o algún miembro del hogar ¿tiene una deuda...,discrete


### Hallazgos
- El codebook documenta **331 variables** (412 discretas + 250 continuas en el XML crudo, deduplicadas por archivo F17/F18).
- Existen variables de **ingreso total** (INGRESO_COMPLETO, INGTOTOB) y componentes (P6050 parentesco, P60xx ingresos laborales).
- La IEFIC complementa los datasets de cartera: permite contextualizar la **capacidad de pago** de los hogares, aunque no hay microdatos en este archivo.